# Yoga Recommendation Engine

Feature-vector based recommendation: each user and each pose is represented as a numeric vector. Suitability scores are computed via weighted compatibility functions.

**No if/else goal routing.** The recommendation is fully numeric.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

In [ ]:
# Inspect pose metadata
from ml.recommendation.yoga_metadata import get_all_poses, poses_to_dataframe
df = poses_to_dataframe()
print(f'Total poses: {len(df)}')
print(df[['name','category','difficulty','flexibility_req','strength_req','duration_min']].to_string(index=False))

In [ ]:
# Inspect pose and user feature vectors
import numpy as np
from ml.recommendation.engine import pose_to_vector, user_to_vector, POSE_VECTOR_DIM

from ml.recommendation.yoga_metadata import YOGA_POSES
sample_pose = YOGA_POSES[0]
pvec = pose_to_vector(sample_pose)
print(f'Pose: {sample_pose.name}')
print(f'Feature vector ({POSE_VECTOR_DIM}-dim): {pvec}')

sample_user = {
    'yoga_experience': 1,
    'flexibility_score': 3.0,
    'strength_score': 2.5,
    'activity_level': 1,
    'session_duration_min': 30.0,
    'primary_goal': 'Flexibility'
}
uvec = user_to_vector(sample_user)
print(f'\nUser vector ({POSE_VECTOR_DIM}-dim): {uvec}')

In [ ]:
# Full recommendation for a sample user
from ml.recommendation.engine import recommend_poses

user = {
    'yoga_experience': 1,
    'flexibility_score': 2.0,
    'strength_score': 2.5,
    'activity_level': 1,
    'session_duration_min': 30.0,
    'primary_goal': 'Flexibility',
    'prefers_restorative': 1
}

recs = recommend_poses(user, top_n=10)
print(f'Top {len(recs)} recommendations for goal=Flexibility, experience=Beginner:\n')
for i, r in enumerate(recs, 1):
    print(f"{i:2d}. [{r.suitability_score:.3f}] {r.pose_name:<35} ({r.difficulty})")
    print(f"        {r.reason}")

In [ ]:
# Compare recommendations for different user profiles
from ml.recommendation.engine import recommend_poses
import pandas as pd

profiles = {
    'Beginner Flexibility': {'yoga_experience':0, 'flexibility_score':2, 'strength_score':2, 'activity_level':0, 'session_duration_min':20, 'primary_goal':'Flexibility'},
    'Intermediate Strength': {'yoga_experience':2, 'flexibility_score':3, 'strength_score':4, 'activity_level':2, 'session_duration_min':45, 'primary_goal':'Strength'},
    'Advanced Mobility': {'yoga_experience':3, 'flexibility_score':5, 'strength_score':4, 'activity_level':3, 'session_duration_min':60, 'primary_goal':'Mobility'},
}

for profile_name, user in profiles.items():
    recs = recommend_poses(user, top_n=5)
    print(f'\n--- {profile_name} ---')
    for r in recs:
        print(f"  [{r.suitability_score:.3f}] {r.pose_name} ({r.difficulty})")

In [ ]:
# Suitability score heatmap: users × poses
import numpy as np
import matplotlib.pyplot as plt
from ml.recommendation.engine import pose_to_vector, user_to_vector, compute_suitability
from ml.recommendation.yoga_metadata import YOGA_POSES

users = [
    {'yoga_experience':0,'flexibility_score':1,'strength_score':1,'activity_level':0,'session_duration_min':20,'primary_goal':'Flexibility'},
    {'yoga_experience':1,'flexibility_score':3,'strength_score':3,'activity_level':1,'session_duration_min':30,'primary_goal':'General Fitness'},
    {'yoga_experience':2,'flexibility_score':4,'strength_score':4,'activity_level':2,'session_duration_min':45,'primary_goal':'Strength'},
    {'yoga_experience':3,'flexibility_score':5,'strength_score':5,'activity_level':3,'session_duration_min':60,'primary_goal':'Mobility'},
]
user_labels = ['Beginner','Light','Moderate','Active']
pose_subset = YOGA_POSES[:20]  # First 20 for readability
pose_names = [p.name for p in pose_subset]

matrix = np.array([
    [compute_suitability(user_to_vector(u), pose_to_vector(p)) for p in pose_subset]
    for u in users
])

plt.figure(figsize=(14, 5))
plt.imshow(matrix, cmap='YlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(label='Suitability Score')
plt.xticks(range(len(pose_names)), pose_names, rotation=45, ha='right', fontsize=8)
plt.yticks(range(len(user_labels)), user_labels)
plt.title('Suitability Score Matrix: Users × Poses')
plt.tight_layout()
plt.show()

## How the Recommendation Works

1. **Pose vector** (15-dim): difficulty, flexibility_req, strength_req, experience_req, duration, category flags (4), goal flags (6)
2. **User vector** (15-dim): headroom for difficulty, capability scores, preference flags, goal one-hot
3. **Suitability** = weighted sum of per-dimension compatibility scores
4. Dimensions with higher weight: goal alignment (2.5×), difficulty/flexibility/strength/experience (1.5–2×)

This is a content-based filtering approach — no collaborative filtering yet (requires real user interaction data).